# Frame2App Full Test Notebook

This notebook tests the `frame2app` package after local installation.

Before running this notebook, make sure the package is installed into the same Python environment used by Jupyter:

```bash
"c:\Users\knowl\anaconda3\envs\clean_env\python.exe" -m pip install -e "C:\Users\knowl\Downloads\frame2app_package\frame2app_package"
```

Then restart the Jupyter kernel and run the cells below in order.

## 1. Environment and package import test

In [ ]:
import sys
print("Python executable:")
print(sys.executable)

import pandas as pd
from frame2app import AutoForm
import frame2app

print("Frame2App imported successfully")
print("Frame2App version:", frame2app.__version__)

## 2. Create sample dataset

This dataset contains integer, float, categorical, boolean, and text columns.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "age": [18, 25, 32, 45, 60, 72],
    "income": [20000, 35000, 50000, 70000, 90000, 120000],
    "gender": ["Male", "Female", "Female", "Male", "Female", "Male"],
    "city": ["London", "Dublin", "Paris", "London", "Berlin", "Paris"],
    "has_account": [True, False, True, True, False, True],
    "risk_score": [0.1, 0.3, 0.6, 0.8, 0.4, 0.9],
    "comments": [
        "new user",
        "regular user",
        "premium user",
        "inactive user",
        "returning user",
        "high value user"
    ],
    "target": [0, 1, 0, 1, 1, 0]
})

df

## 3. Test fully automatic GUI

Expected result: the package automatically creates widgets for all columns except `target`.

In [ ]:
def submit_auto(values):
    print("Automatic form values:")
    print(values)

app_auto = AutoForm(
    data=df,
    target="target",
    title="Test 1: Fully Automatic GUI",
    fields="auto",
    buttons={
        "Submit": submit_auto,
        "Show Values": "show_values",
        "As DataFrame": "to_dataframe",
        "Reset": "reset",
        "Clear Output": "clear_output"
    }
)

app_auto.display()

## 4. Test hybrid automatic + custom GUI

Expected result: most columns are automatic, but `age`, `gender`, `city`, and `comments` use custom widget settings.

In [ ]:
def submit_hybrid(values):
    print("Hybrid form values:")
    print(values)

app_hybrid = AutoForm(
    data=df,
    target="target",
    title="Test 2: Hybrid Auto + Custom GUI",
    fields="auto",
    overrides={
        "age": {
            "widget": "int_slider",
            "min": 18,
            "max": 100,
            "step": 1,
            "default": 30
        },
        "gender": {
            "widget": "radio",
            "options": ["Male", "Female"],
            "default": "Female"
        },
        "city": {
            "widget": "dropdown",
            "options": ["London", "Dublin", "Paris", "Berlin", "New York"],
            "default": "London"
        },
        "comments": {
            "widget": "textarea",
            "default": "",
            "placeholder": "Write user notes here"
        }
    },
    buttons={
        "Run Custom Function": submit_hybrid,
        "Show Values": "show_values",
        "As DataFrame": "to_dataframe",
        "Reset": "reset",
        "Clear Output": "clear_output"
    },
    layout="grid"
)

app_hybrid.display()

## 5. Test fully manual GUI

Expected result: only the fields defined in `fields={...}` appear.

In [ ]:
def submit_manual(values):
    print("Manual form values:")
    print(values)

app_manual = AutoForm(
    data=df,
    title="Test 3: Fully Manual GUI",
    fields={
        "age": {
            "widget": "int_slider",
            "min": 18,
            "max": 100,
            "step": 1,
            "default": 40
        },
        "income": {
            "widget": "int_text",
            "default": 50000
        },
        "gender": {
            "widget": "radio",
            "options": ["Male", "Female"],
            "default": "Male"
        },
        "city": {
            "widget": "dropdown",
            "options": ["London", "Dublin", "Paris", "Berlin"],
            "default": "Paris"
        },
        "has_account": {
            "widget": "checkbox",
            "default": True
        },
        "comments": {
            "widget": "textarea",
            "default": "Testing manual form",
            "placeholder": "Enter comments"
        }
    },
    buttons={
        "Process": submit_manual,
        "Show Values": "show_values",
        "As DataFrame": "to_dataframe",
        "Reset": "reset"
    }
)

app_manual.display()

## 6. Test flexible button names and custom callbacks

Expected result: each button name can be anything, and each button can call a different function.

In [ ]:
def save_record(values):
    print("Saving record...")
    print(values)

def generate_report(values):
    print("Generating report...")
    report = pd.DataFrame([values])
    display(report.describe(include="all"))

def call_api(values):
    print("Pretending to call an API with:")
    print(values)

app_buttons = AutoForm(
    data=df,
    target="target",
    title="Test 4: Flexible Buttons",
    fields="auto",
    buttons={
        "Save Record": save_record,
        "Generate Report": generate_report,
        "Call API": call_api,
        "Show Values": "show_values",
        "Reset": "reset",
        "Clear Output": "clear_output"
    }
)

app_buttons.display()

## 7. Test excluding fields

Expected result: `comments` and `target` do not appear.

In [ ]:
def submit_exclude(values):
    print("Form values without excluded columns:")
    print(values)

app_exclude = AutoForm(
    data=df,
    fields="auto",
    exclude=["comments", "target"],
    title="Test 5: Exclude Fields",
    buttons={
        "Submit": submit_exclude,
        "Show Values": "show_values",
        "Reset": "reset"
    }
)

app_exclude.display()

## 8. Test export helpers

Run this after displaying `app_hybrid` above. Change widget values first if you want to verify live export.

In [ ]:
print("Dictionary output:")
print(app_hybrid.get_values())

print("DataFrame output:")
display(app_hybrid.to_dataframe())

if hasattr(app_hybrid, "to_json"):
    print("JSON output:")
    print(app_hybrid.to_json())
else:
    print("to_json() is not available in this installed version.")

## 9. Test fake prediction workflow

Expected result: button reads GUI values and runs arbitrary user logic.

In [ ]:
def fake_predict(values):
    input_df = pd.DataFrame([values])

    print("Input sent to model:")
    display(input_df)

    if values["age"] > 40 and values["income"] > 50000:
        print("Prediction: High Risk")
    else:
        print("Prediction: Low Risk")

app_predict = AutoForm(
    data=df,
    target="target",
    title="Test 6: Fake Prediction App",
    fields="auto",
    overrides={
        "age": {
            "widget": "int_slider",
            "min": 18,
            "max": 100,
            "default": 45
        },
        "income": {
            "widget": "int_slider",
            "min": 0,
            "max": 150000,
            "step": 5000,
            "default": 60000
        }
    },
    buttons={
        "Predict": fake_predict,
        "Show Input": "to_dataframe",
        "Reset": "reset",
        "Clear Output": "clear_output"
    }
)

app_predict.display()

## 10. Test layout modes

In [ ]:
app_vertical = AutoForm(
    data=df,
    target="target",
    title="Vertical Layout",
    layout="vertical",
    buttons={
        "Submit": lambda values: print(values)
    }
)

app_vertical.display()

In [ ]:
app_grid = AutoForm(
    data=df,
    target="target",
    title="Grid Layout",
    layout="grid",
    buttons={
        "Submit": lambda values: print(values)
    }
)

app_grid.display()

In [ ]:
app_horizontal = AutoForm(
    data=df,
    target="target",
    title="Horizontal Layout",
    layout="horizontal",
    buttons={
        "Submit": lambda values: print(values)
    }
)

app_horizontal.display()

## 11. Test widget type overrides

Expected result: explicit widget types are used instead of automatic inference.

In [ ]:
app_widget_types = AutoForm(
    data=df,
    title="Test 7: Widget Type Overrides",
    fields={
        "age": {
            "widget": "int_slider",
            "min": 0,
            "max": 100,
            "default": 25
        },
        "income": {
            "widget": "int_text",
            "default": 50000
        },
        "risk_score": {
            "widget": "float_slider",
            "min": 0.0,
            "max": 1.0,
            "step": 0.01,
            "default": 0.5
        },
        "gender": {
            "widget": "dropdown",
            "options": ["Male", "Female"],
            "default": "Male"
        },
        "city": {
            "widget": "radio",
            "options": ["London", "Dublin", "Paris", "Berlin"],
            "default": "London"
        },
        "has_account": {
            "widget": "checkbox",
            "default": False
        },
        "comments": {
            "widget": "textarea",
            "default": "",
            "placeholder": "Write something..."
        }
    },
    buttons={
        "Test Widgets": lambda values: print(values),
        "Reset": "reset"
    }
)

app_widget_types.display()

## 12. Test validation behavior

This test assumes your package version includes validation support. If the installed version does not support it, the button may show an unsupported action error.

In [ ]:
def positive_income_validator(value):
    if value < 0:
        return False, "Income cannot be negative."
    return True, ""

def submit_validated(values):
    print("Validated values:")
    print(values)

app_validation = AutoForm(
    data=df,
    title="Test 8: Validation",
    fields={
        "age": {
            "widget": "int_slider",
            "min": 0,
            "max": 120,
            "default": 25,
            "required": True
        },
        "income": {
            "widget": "int_text",
            "default": 50000,
            "validator": positive_income_validator
        },
        "comments": {
            "widget": "textarea",
            "default": "",
            "required": True
        }
    },
    buttons={
        "Validate": "validate",
        "Submit": submit_validated,
        "Reset": "reset"
    }
)

app_validation.display()

## 13. Test real scikit-learn style integration

This trains a simple pipeline and connects it to a Frame2App button.

In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    print("scikit-learn is available")
except ImportError:
    print("scikit-learn is not installed. Run the next cell to install it, then restart the kernel if needed.")

In [ ]:
# Run this only if scikit-learn is missing.
# import sys
# !"{sys.executable}" -m pip install scikit-learn

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop(columns=["target"])
y = df["target"]

categorical_features = ["gender", "city", "has_account", "comments"]
numeric_features = ["age", "income", "risk_score"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

model.fit(X, y)

print("Model trained successfully")

In [ ]:
def real_predict(values):
    input_df = pd.DataFrame([values])
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0]

    print("Input:")
    display(input_df)
    print("Prediction:", prediction)
    print("Probability:", probability)

app_real_model = AutoForm(
    data=df,
    target="target",
    title="Test 9: Real Model Prediction",
    fields="auto",
    overrides={
        "comments": {
            "widget": "dropdown",
            "options": df["comments"].tolist(),
            "default": df["comments"].iloc[0]
        }
    },
    buttons={
        "Predict": real_predict,
        "Show Input": "to_dataframe",
        "Reset": "reset",
        "Clear Output": "clear_output"
    }
)

app_real_model.display()

## 14. Test minimal one-command style

In [ ]:
AutoForm(
    data=df,
    target="target",
    buttons={
        "Run": lambda values: print(values)
    }
).display()

## 15. Test package metadata

In [ ]:
import frame2app
print("Frame2App version:", frame2app.__version__)
print("AutoForm class:", AutoForm)

# End of tests

If all GUI cells display and buttons work, the local package installation is successful.